In [3]:
import math
import os
from pathlib import Path
from datasets import load_dataset
from tqdm.std import tqdm  # Using standard tqdm to avoid ipywidgets warning


def fetch_balanced_dogs_sharded(
    output_dir: str = "images/dog",
    total_target: int = 10000,
    start_label: int = 151,
    end_label: int = 268,
    hf_token: str | None = None,  # Optional: "hf_xxxxxxxx..."
):
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    num_dog_classes = end_label - start_label + 1
    max_per_class = math.ceil(total_target / num_dog_classes)

    class_counts = {
        label_idx: 0 for label_idx in range(start_label, end_label + 1)
    }

    total_saved = 0
    pbar = tqdm(total=total_target, desc="Downloading Dog Images")
    labels = None

    # ImageNet 128x128 currently has 13 training shards (train-00000 to train-00012).
    num_train_shards = 13
    for shard_idx in range(num_train_shards):
        if total_saved >= total_target:
            break

        shard_file = f"data/train-{shard_idx:05d}-of-{num_train_shards:05d}.parquet"
        print(f"\n[Shard {shard_idx + 1}/{num_train_shards}] Downloading '{shard_file}'...")

        # Direct shard download bypasses stream freezing
        shard_ds = load_dataset(
            "benjamin-paine/imagenet-1k-128x128",
            data_files=shard_file,
            split="train",
            token=hf_token,
        )

        if labels is None and "label" in shard_ds.features:
            labels = shard_ds.features["label"].names

        for sample in shard_ds:
            if total_saved >= total_target:
                break

            label_idx = sample["label"]

            if label_idx in class_counts:
                if class_counts[label_idx] < max_per_class:
                    image = sample["image"]

                    if image.mode != "RGB":
                        image = image.convert("RGB")

                    breed_name = (
                        labels[label_idx].split(",")[0].replace(" ", "_")
                        if labels
                        else f"class_{label_idx}"
                    )

                    filename = f"{breed_name}_{class_counts[label_idx]:03d}_{total_saved:05d}.jpg"
                    image.save(out_path / filename, "JPEG", quality=95)

                    class_counts[label_idx] += 1
                    total_saved += 1
                    pbar.update(1)

    pbar.close()
    print(
        f"\nDone! Successfully saved {total_saved} dog images into '{out_path.resolve()}'."
    )


if __name__ == "__main__":
    fetch_balanced_dogs_sharded(
        output_dir="images/dog",
        total_target=10000,
        start_label=151,
        end_label=268,
        hf_token=os.getenv("HF_TOKEN"),  # Optional; public dataset works without a token
    )


[Shard 1/13] Downloading 'data/train-00000-of-00013.parquet'...


















Generating train split:   8%|▊         | 98552/1281167 [00:00<00:09, 122317.86 examples/s]


ExpectedMoreSplitsError: {'validation', 'test'}